In [2]:
!pip install tensorflow

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import numpy as np
print('Packages installed Successfully ✅')

Packages installed Successfully ✅


**Loading Data**

In [5]:
df = pd.read_csv('heart_disease_dataset.csv')
print('Data import successfuly ✅')
df.head()

Data import successfuly ✅


,Age,Gender,Cholesterol,Blood Pressure,Heart Rate,Smoking,Alcohol Intake,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Chest Pain Type,Heart Disease
0,75,Female,228,119,66,Current,Heavy,1,No,No,Yes,8,119,Yes,Atypical Angina,1
1,48,Male,204,165,62,Current,NaN,5,No,No,No,9,70,Yes,Typical Angina,0
2,53,Male,234,91,67,Never,Heavy,3,Yes,No,Yes,5,196,Yes,Atypical Angina,1
3,69,Female,192,90,72,Current,NaN,4,No,Yes,No,7,107,Yes,Non-anginal Pain,0
4,62,Female,172,163,93,Never,NaN,6,No,Yes,No,2,183,Yes,Asymptomatic,0


**EDA**

In [6]:
#Cat to int
df = pd.get_dummies(df, columns=["Gender"],dtype=int)
df = pd.get_dummies(df, columns=["Chest Pain Type"],dtype=int)
df = pd.get_dummies(df, columns=['Smoking'], dtype=int)

#Numeric Columns
num_cols = df.select_dtypes(include=['int64','float64'])
for col in num_cols:
  df[col] = df[col].fillna(df[col].median())

#Cat to int
df['Alcohol Intake'] = df['Alcohol Intake'].fillna(df['Alcohol Intake'].mode()[0])
df = pd.get_dummies(df, columns=['Alcohol Intake'], dtype=int)

#bool to int
boolen_cols = df.select_dtypes(include=['object'])
for col in boolen_cols:
  df = pd.get_dummies(df, columns=[col], dtype=int)

df.head()

,Age,Cholesterol,Blood Pressure,Heart Rate,Exercise Hours,Stress Level,Blood Sugar,Heart Disease,Gender_Female,Gender_Male,...,Alcohol Intake_Heavy,Alcohol Intake_Moderate,Family History_No,Family History_Yes,Diabetes_No,Diabetes_Yes,Obesity_No,Obesity_Yes,Exercise Induced Angina_No,Exercise Induced Angina_Yes
0,75,228,119,66,1,8,119,1,1,0,...,1,0,1,0,1,0,0,1,0,1
1,48,204,165,62,5,9,70,0,0,1,...,1,0,1,0,1,0,1,0,0,1
2,53,234,91,67,3,5,196,1,0,1,...,1,0,0,1,1,0,0,1,0,1
3,69,192,90,72,4,7,107,0,1,0,...,1,0,1,0,0,1,1,0,0,1
4,62,172,163,93,6,2,183,0,1,0,...,1,0,1,0,0,1,1,0,0,1


In [7]:
#Check is there any null
df.isnull().sum()
print( 'No Null or Empty Value in Data set 🔥' if (df.isnull().sum().sum()==0) else 'Null Value present please clear it' )

No Null or Empty Value in Data set 🔥


**Split The Train & Test Data**

In [8]:
X = df.drop(columns=['Heart Disease']).values
y = df['Heart Disease'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')

Shape of X: (1000, 26)
Shape of y: (1000,)


Model Train & Accuracy

In [9]:
model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(16, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0003),
    metrics=["accuracy"]
)
print("\nModel compiled! ✅")

history = model.fit(
    X_train,y_train,epochs=150,batch_size=15, validation_split=0.2,
)
print("\nTraining complete! ✅")

loss, accuracy = model.evaluate(X_test, y_test, batch_size=10, verbose=1)
print(f'Loss value :{loss}')
print(f'Accuracy Value: {accuracy*100:.2f}%')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Model compiled! ✅
Epoch 1/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.4969 - loss: 0.8936 - val_accuracy: 0.5063 - val_loss: 0.7032
Epoch 2/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5078 - loss: 0.8822 - val_accuracy: 0.5813 - val_loss: 0.6877
Epoch 3/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5359 - loss: 0.8392 - val_accuracy: 0.6062 - val_loss: 0.6828
Epoch 4/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5234 - loss: 0.8059 - val_accuracy: 0.6687 - val_loss: 0.6694
Epoch 5/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5406 - loss: 0.7791 - val_accuracy: 0.6250 - val_loss: 0.6646
Epoch 6/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5500 - loss: 0.7639 - val_accuracy: 0.6375 - val_loss: 0.6574
Epoch 7/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6125 - loss: 0.6945 - val_accuracy: 0.6438 - val_loss: 0.6526
Epoch 8/150
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5953 - loss: 0.7255

Predictions on Test Data (First 20 Samples)

In [10]:
predict = model.predict(X_test)

headers = f'{'#':<4} {'Age':>4} {'Age':>8} {"Prob":>8} {'Actual':>10}{'Predict':>10}'
print(headers)
print("-" * len(headers))

X_org = pd.DataFrame(
    scaler.inverse_transform(X_test),
    columns=df.drop(columns=["Heart Disease"]).columns
)

for i in range(20):
  row = X_org.iloc[i]
  prob = predict[i][0]
  gender = "Male" if row["Gender_Male"] == 1 else "Female"
  pred_label   = "Heart Disease 😟"     if prob >= 0.5    else "Happy 😇"
  actual_label = "Heart Disease 😟"     if y_test[i] == 1 else "Happy 😇"
  print(f"{i:<4} {row[0]:>5.0f} {gender:>8} {prob:>8.4f}  {actual_label:>12}  {pred_label:>12}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
#     Age      Age     Prob     Actual   Predict
------------------------------------------------
0       77   Female   0.9737  Heart Disease 😟  Heart Disease 😟
1       42     Male   0.1213       Happy 😇       Happy 😇
2       42   Female   0.0568       Happy 😇       Happy 😇
3       40   Female   0.1210       Happy 😇       Happy 😇
4       78   Female   0.9290  Heart Disease 😟  Heart Disease 😟
5       40   Female   0.0218       Happy 😇       Happy 😇
6       49   Female   0.0682       Happy 😇       Happy 😇
7       43     Male   0.2068       Happy 😇       Happy 😇
8       62     Male   0.6144  Heart Disease 😟  Heart Disease 😟
9       64     Male   0.5608       Happy 😇  Heart Disease 😟
10      68     Male   0.9398  Heart Disease 😟  Heart Disease 😟
11      61     Male   0.6126  Heart Disease 😟  Heart Disease 😟
12      78   Female   0.0560       Happy 😇       Happy 😇
13      65   Female   0.9671  Heart Disease 😟  Heart Disease 😟
14      48   Female   0.590

/tmp/ipykernel_1719/3708668650.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"{i:<4} {row[0]:>5.0f} {gender:>8} {prob:>8.4f}  {actual_label:>12}  {pred_label:>12}")


In [11]:
custome_sample = np.array([
[75,228,119,66,1,8,119,1,1,0,0,1,0,0,1,0,0,1,0,1,0,1,0,0,1,0],
[53,234,91,67,3,5,196,1,0,1,0,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0],
[77,309,110,73,0,4,122,1,0,1,1,0,0,0,0,0,1,1,0,1,0,0,1,0,1,0],
[64,211,105,86,8,2,120,1,1,0,0,0,0,1,0,1,0,1,0,0,1,0,1,0,1,1],
[60,208,148,83,4,2,113,1,1,0,1,0,0,0,0,0,1,0,1,1,0,0,1,0,1,0],
[63,204,141,68,8,3,107,1,0,1,1,0,0,0,0,1,0,1,0,1,0,0,1,1,0,1],
[67,282,108,87,3,9,85,1,1,0,0,0,0,1,0,0,1,1,0,0,1,1,0,0,1,0],
[67,325,177,81,8,10,192,1,0,1,0,0,1,0,0,0,1,0,1,0,1,1,0,0,1,1],
[60,226,168,99,8,10,97,1,0,1,0,0,1,0,0,0,1,0,1,0,1,0,1,1,0,1],
[74,298,148,70,9,8,157,1,1,0,0,0,1,0,0,1,0,1,0,0,1,0,1,0,1,0]
])

persons = [index+1 for index,value in enumerate(custome_sample)]

custome_sample_scaled = scaler.transform(custome_sample)

custom_pred = model.predict(custome_sample_scaled)

print(f"Heart Disease Probability")
print(f"{'Person':>5} {'Probability':>14} {'Status':>15}")
print("-" * 40)

for label, prob in zip(persons, custom_pred):
    status =  "Heart Disease 😟"  if prob >= 0.5  else "Happy 😇"
    print(f"{label:>5} {prob[0]:>14.4f} {status:>15}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Heart Disease Probability
Person    Probability          Status
----------------------------------------
    1         0.6421 Heart Disease 😟
    2         0.4785         Happy 😇
    3         0.9979 Heart Disease 😟
    4         0.7792 Heart Disease 😟
    5         0.1062         Happy 😇
    6         0.3992         Happy 😇
    7         0.9863 Heart Disease 😟
    8         0.9692 Heart Disease 😟
    9         0.0630         Happy 😇
   10         0.9776 Heart Disease 😟
